# Whisper Turbo Transcriber - Google Colab

Notebook ini menjalankan transkripsi audio/video langsung di Google Colab tanpa Streamlit.

Alur penggunaan:

1. Aktifkan GPU: **Runtime -> Change runtime type -> T4 GPU**.
2. Jalankan cell dari atas ke bawah.
3. Upload file audio/video.
4. Tunggu transkripsi selesai.
5. Download ZIP berisi TXT, JSON, dan SRT.

Default model memakai `turbo`, cocok untuk Colab GPU. Kalau GPU sedang tidak tersedia, ganti ke `tiny`, `base`, atau `small`.

## 1. Setup

Cell ini menginstall `ffmpeg`, OpenAI Whisper, dan dependency yang dibutuhkan. Jalankan sekali setiap sesi Colab baru.

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y ffmpeg -qq
!pip install -q openai-whisper torch tqdm

## 2. Helper

Cell ini menyiapkan fungsi export TXT/JSON/SRT/ZIP dan cleanup transkrip sederhana.

In [ ]:
import json
import re
import shutil
import zipfile
from pathlib import Path

import torch
import whisper
from google.colab import files
from tqdm.auto import tqdm

OUTPUT_DIR = Path('outputs')
UPLOAD_DIR = Path('uploads')
OUTPUT_DIR.mkdir(exist_ok=True)
UPLOAD_DIR.mkdir(exist_ok=True)

LANGUAGE_MAP = {
    'Auto-detect': None,
    'Indonesia': 'id',
    'English': 'en',
    'Jawa': 'jw',
    'Sunda': 'su',
    'Melayu': 'ms',
}

FILLER_WORDS = {'anu', 'eee', 'eh', 'em', 'hmm', 'kayak', 'mmm', 'uh', 'umm'}


def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()


def remove_filler_words(text):
    pattern = r'\b(' + '|'.join(re.escape(word) for word in FILLER_WORDS) + r')\b'
    text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+([,.!?])', r'\1', text)
    return normalize_whitespace(text)


def sentence_case(text):
    parts = re.split(r'([.!?]\s+)', text)
    output = []
    for part in parts:
        if not part or re.match(r'[.!?]\s+', part):
            output.append(part)
            continue
        output.append(part[:1].upper() + part[1:])
    result = ''.join(output)
    return result[:1].upper() + result[1:] if result else result


def ensure_sentence_end(text):
    return f'{text}.' if text and text[-1] not in '.!?' else text


def to_paragraphs(text, sentences_per_paragraph=3):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    paragraphs = [
        ' '.join(sentences[index:index + sentences_per_paragraph]).strip()
        for index in range(0, len(sentences), sentences_per_paragraph)
    ]
    return '\n\n'.join(paragraph for paragraph in paragraphs if paragraph)


def minutes_format(text):
    paragraphs = to_paragraphs(text, sentences_per_paragraph=2).split('\n\n')
    bullets = '\n'.join(f'- {paragraph}' for paragraph in paragraphs if paragraph)
    return f'Catatan:\n{bullets}' if bullets else text


def clean_transcript(text, mode):
    if mode == 'Original':
        return text
    cleaned = ensure_sentence_end(sentence_case(remove_filler_words(text)))
    if mode == 'Paragraphs':
        return to_paragraphs(cleaned)
    if mode == 'Minutes':
        return minutes_format(cleaned)
    return cleaned


def format_srt_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    whole_seconds = int(seconds % 60)
    milliseconds = int((seconds - int(seconds)) * 1000)
    return f'{hours:02d}:{minutes:02d}:{whole_seconds:02d},{milliseconds:03d}'


def build_srt(segments):
    lines = []
    for index, segment in enumerate(segments, 1):
        lines.append(str(index))
        lines.append(f"{format_srt_time(segment['start'])} --> {format_srt_time(segment['end'])}")
        lines.append(segment['text'])
        lines.append('')
    return '\n'.join(lines)


def save_outputs(result):
    base = Path(result['file']).stem
    transcript = result.get('cleaned_text') or result['text']
    file_dir = OUTPUT_DIR / base
    file_dir.mkdir(parents=True, exist_ok=True)

    txt_path = file_dir / f'{base}_transcript.txt'
    json_path = file_dir / f'{base}_transcript.json'
    srt_path = file_dir / f'{base}_transcript.srt'

    txt_path.write_text(transcript, encoding='utf-8')
    json_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    srt_path.write_text(build_srt(result['segments']), encoding='utf-8')
    prompt_path = save_ai_prompt(result)
    return txt_path, json_path, srt_path, prompt_path


AI_PROMPT_TEMPLATE = """Tolong rapikan transkrip berikut tanpa mengubah makna.

Instruksi:
- Perbaiki tanda baca, kapitalisasi, dan typo ringan.
- Buat paragraf yang rapi dan mudah dibaca.
- Pertahankan istilah teknis, nama orang, nama tempat, dan angka apa adanya jika tidak yakin.
- Jangan menambah informasi baru di luar transkrip.
- Jika ada bagian yang tidak jelas, tandai dengan [kurang jelas].
- Setelah transkrip rapi, buat ringkasan singkat dan daftar poin penting.

Transkrip:
{transcript}
"""


AI_MINUTES_PROMPT_TEMPLATE = """Ubah transkrip berikut menjadi catatan rapat yang rapi.

Format output:
1. Ringkasan singkat
2. Poin pembahasan utama
3. Keputusan
4. Action items, sertakan PIC dan deadline jika disebutkan
5. Pertanyaan atau bagian yang masih kurang jelas

Aturan:
- Jangan mengarang informasi yang tidak ada di transkrip.
- Kalau PIC/deadline tidak disebutkan, tulis "tidak disebutkan".
- Pertahankan bahasa utama transkrip.

Transkrip:
{transcript}
"""


def build_ai_prompt(transcript, mode='clean'):
    template = AI_MINUTES_PROMPT_TEMPLATE if mode == 'minutes' else AI_PROMPT_TEMPLATE
    return template.format(transcript=transcript)



def save_ai_prompt(result):
    base = Path(result['file']).stem
    transcript = result.get('cleaned_text') or result['text']
    file_dir = OUTPUT_DIR / base
    file_dir.mkdir(parents=True, exist_ok=True)

    prompt_mode = 'minutes' if result.get('cleanup_mode') == 'Minutes' else 'clean'
    prompt = build_ai_prompt(transcript, prompt_mode)
    prompt_path = file_dir / f'{base}_ai_prompt.md'
    prompt_path.write_text(prompt, encoding='utf-8')
    return prompt_path


def make_zip(zip_name='transcripts.zip'):
    zip_path = Path(zip_name)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in OUTPUT_DIR.rglob('*'):
            if path.is_file():
                archive.write(path, path.relative_to(OUTPUT_DIR))
    return zip_path


print('Helper ready.')

## 3. Pengaturan Transkripsi

Ubah nilai di bawah sesuai kebutuhan. Untuk Colab GPU, `turbo` adalah pilihan default yang direkomendasikan.

In [ ]:
MODEL_SIZE = 'turbo'  #@param ['tiny', 'base', 'small', 'medium', 'turbo', 'large']
LANGUAGE_LABEL = 'Indonesia'  #@param ['Auto-detect', 'Indonesia', 'English', 'Jawa', 'Sunda', 'Melayu']
TASK = 'transcribe'  #@param ['transcribe', 'translate']
CLEANUP_MODE = 'Paragraphs'  #@param ['Original', 'Clean only', 'Paragraphs', 'Minutes']

LANGUAGE = LANGUAGE_MAP[LANGUAGE_LABEL]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')
print(f'Model: {MODEL_SIZE}')
print(f'Language: {LANGUAGE_LABEL}')
print(f'Task: {TASK}')
print(f'Cleanup: {CLEANUP_MODE}')

if DEVICE != 'cuda' and MODEL_SIZE in {'medium', 'turbo', 'large'}:
    print('\nPERINGATAN: GPU tidak aktif. Model besar akan sangat lambat di CPU.')
    print('Aktifkan Runtime -> Change runtime type -> T4 GPU, atau ganti model ke tiny/base/small.')

## 4. Upload File

Jalankan cell ini, lalu pilih satu atau beberapa file audio/video dari laptop.

In [ ]:
uploaded = files.upload()

uploaded_paths = []
for file_name, content in uploaded.items():
    target = UPLOAD_DIR / file_name
    target.write_bytes(content)
    uploaded_paths.append(target)

print(f'{len(uploaded_paths)} file uploaded:')
for path in uploaded_paths:
    print(f'- {path}')

## 5. Jalankan Transkripsi

Cell ini akan load model, memproses semua file, menyimpan hasil ke folder `outputs/`, lalu membuat `transcripts.zip`.

In [ ]:
if not uploaded_paths:
    raise RuntimeError('Belum ada file. Jalankan cell upload terlebih dahulu.')

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Loading Whisper model: {MODEL_SIZE} on {DEVICE}...')
model = whisper.load_model(MODEL_SIZE, device=DEVICE)
print('Model ready.\n')

results = []
options = {'task': TASK}
if LANGUAGE:
    options['language'] = LANGUAGE

for audio_path in tqdm(uploaded_paths, desc='Transcribing files'):
    print(f'\nTranscribing: {audio_path.name}')
    raw = model.transcribe(str(audio_path), **options)
    text = raw['text'].strip()
    segments = [
        {
            'id': segment['id'],
            'start': round(segment['start'], 2),
            'end': round(segment['end'], 2),
            'text': segment['text'].strip(),
        }
        for segment in raw.get('segments', [])
    ]
    cleaned_text = clean_transcript(text, CLEANUP_MODE)
    result = {
        'file': audio_path.name,
        'model': MODEL_SIZE,
        'language': raw.get('language', 'unknown'),
        'task': TASK,
        'cleanup_mode': CLEANUP_MODE,
        'text': text,
        'cleaned_text': cleaned_text,
        'segments': segments,
    }
    save_outputs(result)
    results.append(result)

zip_path = make_zip()
print(f'\nDone. ZIP ready: {zip_path}')

## 6. Preview Hasil

Cell ini menampilkan ringkasan dan sebagian transkrip pertama.

In [ ]:
for result in results:
    transcript = result.get('cleaned_text') or result['text']
    duration = result['segments'][-1]['end'] if result['segments'] else 0
    print('=' * 80)
    print(f"File      : {result['file']}")
    print(f"Language  : {result['language']}")
    print(f"Duration  : {duration:.2f} sec")
    print(f"Words     : {len(transcript.split())}")
    print(f"Segments  : {len(result['segments'])}")
    print('-' * 80)
    print(transcript[:2000])
    if len(transcript) > 2000:
        print('\n... preview dipotong. Download ZIP untuk teks lengkap.')


## 7. Lanjut Rapikan dengan AI

Gunakan cell ini kalau ingin menyempurnakan hasil di ChatGPT, Claude, Gemini, Copilot, atau AI lain. Prompt juga otomatis disimpan ke ZIP sebagai file `_ai_prompt.md` per transkrip.


In [ ]:
if not results:
    raise RuntimeError('Belum ada hasil transkripsi. Jalankan transkripsi dulu.')

print('Prompt lengkap untuk setiap file sudah disimpan di ZIP:')
for result in results:
    base = Path(result['file']).stem
    print(f'- outputs/{base}/{base}_ai_prompt.md')

first_result = results[0]
first_transcript = first_result.get('cleaned_text') or first_result['text']
prompt_mode = 'minutes' if first_result.get('cleanup_mode') == 'Minutes' else 'clean'
ai_prompt = build_ai_prompt(first_transcript, prompt_mode)

print('\nPreview prompt file pertama untuk ChatGPT / Claude / Gemini / Copilot:')
print('=' * 80)
print(ai_prompt[:6000])
if len(ai_prompt) > 6000:
    print('\n... prompt dipotong di preview. Pakai file *_ai_prompt.md di ZIP untuk prompt lengkap.')


## 8. Download ZIP

ZIP berisi folder per file dengan output:

- `.txt` untuk transkrip bersih
- `.json` untuk data lengkap + timestamp
- `.srt` untuk subtitle

In [ ]:
if 'zip_path' not in globals() or not Path(zip_path).exists():
    raise RuntimeError('ZIP belum tersedia. Jalankan cell transkripsi terlebih dahulu.')

files.download(str(zip_path))


## Tips

- Kalau `turbo` lambat atau crash, pastikan GPU aktif di **Runtime -> Change runtime type -> T4 GPU**.
- Untuk file sangat panjang, potong audio menjadi beberapa bagian agar lebih stabil.
- Gunakan `translate` kalau ingin output English dari audio bahasa lain.
- Gunakan cleanup `Original` kalau ingin hasil mentah Whisper tanpa perubahan teks.